In [139]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score
)
from sklearn.neighbors import NearestNeighbors

from sklearn.decomposition import PCA


In [140]:
BASE_DIR = Path()
DATA_DIR = BASE_DIR / "data"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
FIGURES_DIR = ARTIFACTS_DIR / "figures"
LABELS_DIR = ARTIFACTS_DIR / "labels"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
LABELS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = {
    "ds1": DATA_DIR / "S07-hw-dataset-01.csv",
    "ds2": DATA_DIR / "S07-hw-dataset-02.csv",
    "ds3": DATA_DIR / "S07-hw-dataset-03.csv",
}

metrics_summary = {}
best_configs = {}


In [141]:
def build_preprocessor(df: pd.DataFrame):
    num_cols = df.select_dtypes(include=["number"]).columns.tolist()
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), num_cols)
        ]
    )
    return preprocessor, num_cols


def compute_metrics(X, labels):
    # внутренние метрики качества кластеризации
    mask = labels != -1
    X_eval = X[mask]
    labels_eval = labels[mask]

    n_clusters = len(set(labels_eval))
    noise_ratio = float((labels == -1).mean())
    # Критические условия для некорректной кластеризации
    if n_clusters < 2:
        return {
            "silhouette": -1,
            "davies_bouldin": float('inf'),
            "calinski_harabasz": -1,
            "noise_ratio": noise_ratio,
        }
    
    return {
        "silhouette": silhouette_score(X_eval, labels_eval),
        "davies_bouldin": davies_bouldin_score(X_eval, labels_eval),
        "calinski_harabasz": calinski_harabasz_score(X_eval, labels_eval),
        "noise_ratio": noise_ratio,
    }


def pca_scatter(X, labels, title, path):
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X)

    plt.figure(figsize=(6, 5))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, s=12, cmap="tab10")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(path)
    plt.close()

def is_dbscan_better(db, km):
    return (
        db["silhouette"] > km["silhouette"] + 0.1
        and db["davies_bouldin"] < km["davies_bouldin"]
        and db["noise_ratio"] < 0.3
    )


Основная программа

In [142]:
for ds_name, ds_path in DATASETS.items():
    print(f"\n{'='*50}\nОбработка: {ds_name}\n{'='*50}")

    # 1. Загрузка данных и первичный анализ
    df = pd.read_csv(ds_path)
    print("\nПервые 5 строк:")
    display(df.head())
    print("\nИнформация о данных:")
    display(df.info())
    print("\nОписательная статистика:")
    display(df.describe())

    # 2. Препроцессинг
    sample_id = df["sample_id"]
    X_raw = df.drop(columns=["sample_id"])

    preprocessor, num_cols = build_preprocessor(X_raw)
    X = preprocessor.fit_transform(X_raw)
    print(f"\nПрепроцессинг завершен.")

    # 3. KMeans - подбор оптимального k
    print("\nЗапуск KMeans...")
    k_range = range(2, 21)
    km_results = {}

    silhouettes = []

    for k in k_range:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X)
        sil = silhouette_score(X, labels)
        silhouettes.append(sil)

        km_results[k] = {
            "labels": labels,
            "metrics": {
                "silhouette": sil,
                "davies_bouldin": davies_bouldin_score(X, labels),
                "calinski_harabasz": calinski_harabasz_score(X, labels),
            }
        }

    best_k = max(km_results, key=lambda k: km_results[k]["metrics"]["silhouette"])
    best_km = km_results[best_k]
    print(f"Лучшее k для KMeans: {best_k} (Silhouette = {best_km['metrics']['silhouette']:.4f})")
    
    # Визуализация подбора k
    plt.figure()
    plt.plot(list(k_range), silhouettes, marker="o")
    plt.xlabel("k")
    plt.ylabel("silhouette")
    plt.title(f"{ds_name} — KMeans silhouette vs k")
    plt.savefig(FIGURES_DIR / f"{ds_name}_kmeans_silhouette.png")
    plt.close()

    """
     # 4. DBSCAN - подбор параметров
    print("\nЗапуск DBSCAN...")
    dbscan = DBSCAN(eps=0.5, min_samples=1)
    db_labels = dbscan.fit_predict(X)

    db_metrics = compute_metrics(X, db_labels)
    """
    print("\nЗапуск DBSCAN с автоматическим подбором eps...")
    n_samples = X.shape[0]

    min_samples = max(5, int(np.log(n_samples)))
    print(f"min_samples = {min_samples} (1% от {n_samples})")
    
    
    # Расчет k-дистанций
    nbrs = NearestNeighbors(n_neighbors=min_samples, n_jobs=-1).fit(X)
    distances, _ = nbrs.kneighbors(X)
    k_distances = distances[:, -1]  # расстояния до min_samples-го соседа
    sorted_dists = np.sort(k_distances)
    
    # 1. Нормализация расстояний для устойчивого поиска локтя
    y_norm = (sorted_dists - sorted_dists.min()) / (sorted_dists.max() - sorted_dists.min() + 1e-10)
    x_norm = np.linspace(0, 1, len(y_norm))
    
    # 2. Поиск точки с максимальной кривизной (вторая производная)
    dy = np.gradient(y_norm)
    d2y = np.gradient(dy)
    curvature = np.abs(d2y)
    knee_idx = np.argmax(curvature[100:-100]) + 100  # Игнорируем края
    
    # 3. Резервный метод: 90-й процентиль для монотонных графиков
    if knee_idx < 0.2 * len(sorted_dists) or knee_idx > 0.8 * len(sorted_dists):
        eps = np.percentile(sorted_dists, 70)
        method_used = "70th percentile (резерв)"
    else:
        eps = sorted_dists[knee_idx]
        method_used = "maximum curvature"
    
    # 4. Защита от экстремальных значений
    eps = np.clip(eps, 0.1, 5.0)  # Разумные границы для eps
    
    # Визуализация с маркерами
    plt.figure(figsize=(8, 5))
    plt.plot(range(len(sorted_dists)), sorted_dists, label='k-distance')
    plt.axvline(x=knee_idx, color='r', linestyle='--', label=f'MAX Локоть (idx={knee_idx})')
    plt.axhline(y=eps, color='g', linestyle=':', label=f'eps={eps:.4f}')
    plt.axvline(x=int(0.9 * len(sorted_dists)), color='b', linestyle='-.', label='90%')
    plt.title(f'{ds_name} - k-distance graph ({method_used})')
    plt.xlabel('Точки (отсортированы)')
    plt.ylabel('Расстояние до k-го соседа')
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"{ds_name}_dbscan_kdist.png")
    plt.close()
    
    # --- НАЧАЛО ВСТАВКИ ---
    print(f"Базовый eps={eps:.4f}, проверяем окрестность...")
    
    # Проверяем 5 вариантов eps вокруг базового значения
    best_db_score = -float('inf')
    best_db_params = None
    best_db_labels = None
    best_db_metrics = None
    
    for eps_offset in [-0.2, -0.1, 0, 0.1, 0.2]:
        test_eps = eps + eps_offset
        test_eps = np.clip(test_eps, 0.1, 5.0)
        
        dbscan = DBSCAN(eps=test_eps, min_samples=min_samples)
        labels = dbscan.fit_predict(X)
        metrics = compute_metrics(X, labels)
        if (
            metrics["silhouette"] > best_db_score
            and metrics["noise_ratio"] <= 0.4
        ):
            best_db_score = metrics["silhouette"]
            best_db_params = test_eps
            best_db_labels = labels
            best_db_metrics = metrics
    
    # Используем лучшие параметры из grid search
    eps = best_db_params
    db_labels = best_db_labels
    db_metrics = best_db_metrics
    print(f"Лучший eps={eps:.4f} (score={best_db_score:.4f})")
    # ---------- Выбор лучшего ----------
    best_method = "kmeans"
    best_labels = best_km["labels"]
    best_metrics = best_km["metrics"]
    
    db_is_valid = (
        db_metrics is not None
        and db_metrics["silhouette"] > 0
        and db_metrics["noise_ratio"] <= 0.3
    )
    
    if db_is_valid and is_dbscan_better(db_metrics, best_metrics):
        best_method = "dbscan"
        best_labels = db_labels
        best_metrics = db_metrics
        print(
            f"Выбран DBSCAN: "
            f"sil={db_metrics['silhouette']:.3f}, "
            f"DB={db_metrics['davies_bouldin']:.3f}, "
            f"noise={db_metrics['noise_ratio']:.2f}"
        )
    else:
        print(
            f"KMeans выбран: "
            f"sil={best_metrics['silhouette']:.3f}, "
            f"DB={best_metrics['davies_bouldin']:.3f}"
        )
    pca_scatter(
        X,
        best_labels,
        title=f"{ds_name} — best clustering ({best_method})",
        path=FIGURES_DIR / f"{ds_name}_pca_best.png"
    )
    # 7. Сохранение результатов
    metrics_summary[ds_name] = {
        "kmeans": best_km["metrics"],
        "dbscan": db_metrics,
    }

    best_configs[ds_name] = {
        "method": best_method,
        "params": {
            "k": best_k if best_method == "kmeans" else None,
            "eps": eps if best_method == "dbscan" else None,  # Исправлено!
            "min_samples": min_samples if best_method == "dbscan" else None
        },
        "criterion": "silhouette_score",
    }

    pd.DataFrame({
        "sample_id": sample_id,
        "cluster_label": best_labels
    }).to_csv(LABELS_DIR / f"labels_{ds_name}.csv", index=False)
    if ds_name == "ds1":
        labels_list = []
        for seed in range(5):
            km = KMeans(n_clusters=best_k, random_state=seed, n_init=10)
            labels_list.append(km.fit_predict(X))
    
        ari_scores = []
        for i in range(1, 5):
            ari_scores.append(
                adjusted_rand_score(labels_list[0], labels_list[i])
            )
    
        print("Stability ARI:", ari_scores)


Обработка: ds1

Первые 5 строк:


,sample_id,f01,f02,f03,f04,f05,f06,f07,f08
0,0,-0.536647,-69.812900,-0.002657,71.743147,-11.396498,-12.291287,-6.836847,-0.504094
1,1,15.230731,52.727216,-1.273634,-104.123302,11.589643,34.316967,-49.468873,0.390356
2,2,18.542693,77.317150,-1.321686,-111.946636,10.254346,25.892951,44.595250,0.325893
3,3,-12.538905,-41.709458,0.146474,16.322124,1.391137,2.014316,-39.930582,0.139297
4,4,-6.903056,61.833444,-0.022466,-42.631335,3.107154,-5.471054,7.001149,0.131213



Информация о данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12000 entries, 0 to 11999
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  12000 non-null  int64  
 1   f01        12000 non-null  float64
 2   f02        12000 non-null  float64
 3   f03        12000 non-null  float64
 4   f04        12000 non-null  float64
 5   f05        12000 non-null  float64
 6   f06        12000 non-null  float64
 7   f07        12000 non-null  float64
 8   f08        12000 non-null  float64
dtypes: float64(8), int64(1)
memory usage: 843.9 KB


None


Описательная статистика:


...


Информация о данных:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   sample_id  15000 non-null  int64  
 1   x1         15000 non-null  float64
 2   x2         15000 non-null  float64
 3   f_corr     15000 non-null  float64
 4   f_noise    15000 non-null  float64
dtypes: float64(4), int64(1)
memory usage: 586.1 KB


None


Описательная статистика:


,sample_id,x1,x2,f_corr,f_noise
count,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000
mean,7499.500000,1.246296,1.033764,0.212776,-0.027067
std,4330.271354,4.592421,4.710791,1.530017,2.506375
min,0.000000,-9.995585,-9.980853,-5.212038,-8.785884
25%,3749.750000,-1.782144,-2.666393,-0.966224,-1.731128
50%,7499.500000,0.664226,1.831257,0.296508,-0.052391
75%,11249.250000,4.435671,4.969630,1.390273,1.673831
max,14999.000000,16.207863,14.271153,5.795876,11.266865



Препроцессинг завершен.

Запуск KMeans...
Лучшее k для KMeans: 3 (Silhouette = 0.3155)

Запуск DBSCAN с автоматическим подбором eps...
min_samples = 9 (1% от 15000)
Базовый eps=0.3099, проверяем окрестность...
Лучший eps=0.5099 (score=0.1419)
KMeans выбран: sil=0.316, DB=1.158


Сохранения

In [143]:
with open(ARTIFACTS_DIR / "metrics_summary.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)

with open(ARTIFACTS_DIR / "best_configs.json", "w") as f:
    json.dump(best_configs, f, indent=2)